In [1]:
import json
import os
from pathlib import Path

import Cleaning_Splitting
import torch
import torch.nn as nn
from data_analyzer import *
from huggingface_hub import hf_hub_download, login
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer


In [2]:
MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"

model_path = hf_hub_download(
    repo_id=MODEL_NAME,
    filename="pytorch_model.bin"
)

print("Model downloaded successfully!")
print(model_path)

Model downloaded successfully!
C:\Users\Mahdi\.cache\huggingface\hub\models--HooshvareLab--bert-fa-base-uncased\snapshots\a04aa40c97bcdde570ae11986a534542c2995a62\pytorch_model.bin


In [3]:
OUTPUT_FOLDER = ".."
OUTPUT_NAME = "artifacts"
OUTPUT_DIR = Path(os.path.join(OUTPUT_FOLDER,OUTPUT_NAME))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
find_data()

The current file location :  d:\Projects\python projects\boot_camp_HW\project 3\notebooks
File exists: True
File size: 1.19 GB


In [5]:
df = df_maker()
RANDOM_STATE = 42


print("Original dataset shape:", df.shape)

Original dataset shape: (6156289, 15)


# Analyzing data

In [6]:
# report_missing(df)

# analyze_rate_recommendation(df)

# analyze_body_length(df)

# analyze_duplicates_and_products(df)

# analyze_exact_duplicates(df)

In [7]:
df_clean = df.drop_duplicates(subset="id", keep="first").copy()

In [8]:
df_process = Cleaning_Splitting.create_rating_band(df_clean)

In [9]:
# max_rate = df_process["rate"].max() ==> 2500

df_process.loc[
    df_process["rate"] == 2500,
    "rating_band"
] = "High"

In [10]:
df_preprosses = Cleaning_Splitting.create_processing_df(df_process)

print("Processing dataset shape:", df_preprosses.shape)
print("samples:")
df_preprosses.head()

Processing dataset shape: (6153060, 5)
samples:


,id,product_id,body,recommendation_status,rating_band
0,53672599,252058,به درد نمیخوره,not_recommended,Low
1,9897229,252058,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,recommended,Low
2,38074516,3331597,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خ...,recommended,Low
3,18628562,3331329,به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد ...,recommended,Low
4,53301258,3255700,معمولیه اگه واسه خونه رنگ کردن شخصی میخواین او...,recommended,Medium


In [11]:
df_preprosses = df_preprosses[
    df_preprosses["body"].str.strip().ne("")
].copy()

print("After removing empty reviews:", df_preprosses.shape)

After removing empty reviews: (6153060, 5)


In [12]:
strata_counts = check_split_strata(df_preprosses)

print(strata_counts)

not_recommended__High        14759
not_recommended__Medium      70495
MISSING__Low                 99573
no_idea__Low                100344
MISSING__Medium             105840
no_idea__High               115787
no_idea__Medium             377587
not_recommended__Low        419255
recommended__Medium         439145
recommended__Low            460221
MISSING__High               688657
recommended__High          3261397
Name: count, dtype: int64[pyarrow]


In [13]:
labeled_df, unlabeled_df = Cleaning_Splitting.separate_labeled_data(df_preprosses)

print("Labeled data:", labeled_df.shape)
print("Unlabeled data:", unlabeled_df.shape)

Labeled data: (5258990, 5)
Unlabeled data: (894070, 5)


In [14]:
print("Labeled:")
print(labeled_df.dtypes)

print("\nUnlabeled:")
print(unlabeled_df.dtypes)

print("\nDetailed comparison:")
print(pd.DataFrame({
    "Labeled_dtype": labeled_df.dtypes,
    "Unlabeled_dtype": unlabeled_df.dtypes,
    "Labeled_non_null": labeled_df.notna().sum(),
    "Unlabeled_non_null": unlabeled_df.notna().sum()
}))

Labeled:
id                          int64
product_id                  int64
body                          str
recommendation_status         str
rating_band              category
dtype: object

Unlabeled:
id                          int64
product_id                  int64
body                          str
recommendation_status         str
rating_band              category
dtype: object

Detailed comparison:
                      Labeled_dtype Unlabeled_dtype  Labeled_non_null  \
id                            int64           int64           5258990   
product_id                    int64           int64           5258990   
body                            str             str           5258489   
recommendation_status           str             str           5258990   
rating_band                category        category           5258990   

                       Unlabeled_non_null  
id                                 894070  
product_id                         894070  
body              

In [15]:
max(df_process["rate"])

2500.0

In [16]:
split_seed = 2026
train_df, validation_df, test_df = Cleaning_Splitting.split_by_product(
    labeled_df
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (3756421, 5)
Validation: (751284, 5)
Test: (751285, 5)


In [17]:
location = train_df.iloc[
    train_df["rating_band"].isna().to_numpy().nonzero()[0]
].index
train_df.loc[location, "rating_band"] = "High"

In [18]:
print(f"precentage: {validation_df.__len__()/labeled_df.__len__() }")

precentage: 0.14285708852840565


In [19]:
compare_label_distribution(train_df, validation_df, test_df)

,Train,Validation,Test
recommendation_status,,,
recommended,79.12,79.12,79.12
no_idea,11.29,11.29,11.29
not_recommended,9.59,9.59,9.59


In [20]:
test_df.head(2)

,id,product_id,body,recommendation_status,rating_band
3,18628562,3331329,به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد ...,recommended,Low
22,15789424,2648752,در کل نسبت به پولی که پرداخت میکنید خوبه,recommended,Low


In [21]:
for data in [train_df, validation_df, test_df]:
    data["body_clean"] = data["body"].apply(Cleaning_Splitting.normalize_persian_text)

emoji testing

In [22]:
# # Find reviews containing emojis in original text
# samples = train_df[
#     train_df["body_clean"].apply(
#         lambda x: isinstance(x, str) and emoji.emoji_count(x) > 0
#     )
# ].head(10)

# for _, row in samples.iterrows():

#     print("Original:")
#     print(row["body"])

#     print("\nNormalized:")
#     print(Cleaning_Splitting.normalize_persian_text(row["body"]))

#     print("\nFinal:")
#     print(row["body_clean"])

#     print("=" * 60)

In [23]:
# Define inputs
X_train = train_df["body_clean"]
X_validation = validation_df["body_clean"]
X_test = test_df["body_clean"]

# Define targets
y_train = train_df[
    ["recommendation_status", "rating_band"]
]

y_validation = validation_df[
    ["recommendation_status", "rating_band"]
]

y_test = test_df[
    ["recommendation_status", "rating_band"]
]



for name, X, y in [
    ("Train", X_train, y_train),
    ("Validation", X_validation, y_validation),
    ("Test", X_test, y_test)
]:
    print(f"{name}:")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Missing labels: {y.isna().sum()}")
    print("-" * 30)

Train:
X shape: (3756421,)
y shape: (3756421, 2)
Missing labels: recommendation_status    0
rating_band              0
dtype: int64
------------------------------
Validation:
X shape: (751284,)
y shape: (751284, 2)
Missing labels: recommendation_status    0
rating_band              0
dtype: int64
------------------------------
Test:
X shape: (751285,)
y shape: (751285, 2)
Missing labels: recommendation_status    0
rating_band              0
dtype: int64
------------------------------


In [24]:
MASK_RATE = 0.20
(
    train_public_df,
    train_original_df,
    mask_counts,
    mask_manifest
) = Cleaning_Splitting.apply_public_masking(
    train_df,
    mask_rate=MASK_RATE
)

In [25]:
train_public_df.shape

(3756421, 6)

In [26]:
train_public_df.head(2)

,id,product_id,body,recommendation_status,rating_band,body_clean
0,53672599,252058,به درد نمیخوره,not_recommended,Low,به درد نمیخوره
1,9897229,252058,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,recommended,Low,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...


In [27]:
print("Total masked:", mask_manifest.shape[0])
mask_counts

Total masked: 3756421


,class,total_count,masked_count,remaining_count,mask_rate_actual,rounding
0,no_idea,424084,84816,339268,0.199998,floor
1,not_recommended,360364,72072,288292,0.199998,floor
2,recommended,2971973,594394,2377579,0.200000,floor


In [28]:
train_df.to_csv(OUTPUT_DIR / "train_df.csv", index=False)
train_public_df.to_csv(OUTPUT_DIR / "train_public.csv", index=False)
train_original_df.to_csv(OUTPUT_DIR / "train_original.csv", index=False)
validation_df.to_csv(OUTPUT_DIR / "validation.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)
mask_manifest.to_csv(OUTPUT_DIR / "mask_manifest.csv", index=False)

In [29]:
split_manifest = {
    "split_seed":split_seed,
    "split_strategy": "StratifiedGroupSplit",
    "group_column": "product_id",
    "target_columns": ["recommendation_status", "rating_band"],
    "split_sizes": {
        "train": len(train_df),
        "validation": len(validation_df),
        "test": len(test_df)
    },
    "split_ratios": {
        "train": len(train_df) / len(df_preprosses),
        "validation": len(validation_df) / len(df_preprosses),
        "test": len(test_df) / len(df_preprosses)
    }
}
with open(OUTPUT_DIR / "split_manifest.json", "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, ensure_ascii=False, indent=4)

In [30]:
# Check product leakage
assert set(train_df["product_id"]).isdisjoint(validation_df["product_id"])
assert set(train_df["product_id"]).isdisjoint(test_df["product_id"])
assert set(validation_df["product_id"]).isdisjoint(test_df["product_id"])

# Check row counts
assert len(train_df) + len(validation_df) + len(test_df) == len(labeled_df)
print("All validation checks passed!")

All validation checks passed!


In [31]:
# X_train = train_df["body"]
# X_val = validation_df["body"]
# X_test = test_df["body"]

# y_train = train_df["recommendation_status","rating_band"]
# y_val = validation_df["recommendation_status"]
# y_test = test_df["recommendation_status"]

In [32]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_validation)
X_test_tfidf = tfidf.transform(X_test)

In [33]:
print("Train:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Test:", X_test_tfidf.shape)

Train: (3756421, 20000)
Validation: (751284, 20000)
Test: (751285, 20000)


In [34]:
target = train_df["recommendation_status"]

In [35]:
# location = y_train.iloc[
#     y_train.isna().any(axis=1).to_numpy().nonzero()[0]
# ].index

In [36]:
# df.loc[location]

In [37]:
# y_train.loc[location, "rating_band"] = "High"

In [38]:
# y_train.loc[location]

In [39]:
base_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model = MultiOutputClassifier(base_model)

model.fit(X_train_tfidf, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,LogisticRegre...ndom_state=42)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Class labels.",list,"[array(['no_id... dtype=object), array(['High'... dtype=object)]"
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42)]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,20000
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0


In [40]:
y_predict_val = model.predict(X_val_tfidf)

In [41]:
target_names = ["recommendation_status","rating_band"]

for i, target_name in enumerate(target_names):

    print(f"\n{'='*40}")
    print(f"Target: {target_name}")
    print(f"{'='*40}")

    y_true = y_validation.iloc[:, i]
    y_pred = y_predict_val[:, i]

    print("Accuracy:", accuracy_score(
        y_true, y_pred
    ))

    print("Precision:", precision_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    ))

    print("Recall:", recall_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    ))

    print("F1 Score:", f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    ))


Target: recommendation_status
Accuracy: 0.7934096826233489
Precision: 0.6354547503532232
Recall: 0.7307434521521818
F1 Score: 0.6658295778069299

Target: rating_band
Accuracy: 0.6351952124629301
Precision: 0.5508341408344606
Recall: 0.5779145026103292
F1 Score: 0.5567641040960619


In [64]:
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

ENCODER_LR = 2e-5
HEAD_LR = 1e-4

MAX_LENGTH = 128
EPOCHS = 5

MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [65]:
targets = ["recommendation_status","rating_band"]

target_encoders = {}

for target in targets:
    encoder = LabelEncoder()

    non_null_values = train_df[target].dropna()
    encoder.fit(non_null_values)

    target_encoders[target] = encoder

    print(target)
    print(dict(zip(
        encoder.classes_,
        encoder.transform(encoder.classes_)
    )))

recommendation_status
{'no_idea': np.int64(0), 'not_recommended': np.int64(1), 'recommended': np.int64(2)}
rating_band
{'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}


In [66]:
class MultiTaskDataset(Dataset):

    def __init__(
        self,
        df,
        tokenizer,
        target_encoders,
        max_length=128
    ):
        self.df = df.copy().reset_index(drop=True)
        self.tokenizer = tokenizer
        self.target_encoders = target_encoders
        self.max_length = max_length

        self.targets = list(target_encoders.keys())

        # Keep only samples with all real labels
        self.df = self.df.dropna(
            subset=self.targets
        ).reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        encoding = self.tokenizer(
            str(row["body_clean"]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        labels = [
            int(
                self.target_encoders[target].transform(
                    [row[target]]
                )[0]
            )
            for target in self.targets
        ]

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.long)
        }


In [67]:
train_dataset = MultiTaskDataset(
    train_df,
    tokenizer,
    target_encoders,
    MAX_LENGTH
)

validation_dataset = MultiTaskDataset(
    validation_df,
    tokenizer,
    target_encoders,
    MAX_LENGTH
)

test_dataset = MultiTaskDataset(
    test_df,
    tokenizer,
    target_encoders,
    MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)

In [68]:
num_classes = {
    target: len(encoder.classes_)
    for target, encoder in target_encoders.items()
}

print(num_classes)

{'recommendation_status': 3, 'rating_band': 3}


In [69]:
class MultiTaskBERT(nn.Module):

    def __init__(self, model_name, num_classes, dropout=0.3):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.recommendation_head = nn.Linear(
            hidden_size,
            num_classes["recommendation_status"]
        )

        self.rating_head = nn.Linear(
            hidden_size,
            num_classes["rating_band"]
        )

    def forward(self, input_ids, attention_mask):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled_output = outputs.last_hidden_state[:, 0, :]

        pooled_output = self.dropout(pooled_output)

        recommendation_logits = self.recommendation_head(
            pooled_output
        )

        rating_logits = self.rating_head(
            pooled_output
        )

        return recommendation_logits, rating_logits

In [70]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MultiTaskBERT(
    MODEL_NAME,
    num_classes
).to(device)

print("Device:", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda


In [71]:
batch = next(iter(train_loader))

print("Input IDs:", batch["input_ids"].shape)
print("Attention Mask:", batch["attention_mask"].shape)
print("Labels:", batch["labels"].shape)

print("\nLabels:")
print(batch["labels"])

print("\nDevice:", device)

Input IDs: torch.Size([2, 128])
Attention Mask: torch.Size([2, 128])
Labels: torch.Size([2, 2])

Labels:
tensor([[2, 0],
        [2, 2]])

Device: cuda


In [72]:
class SupervisedDataset(Dataset):

    def __init__(self, df, tokenizer, target_encoders, max_length=128):

        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.target_encoders = target_encoders
        self.targets = list(target_encoders.keys())
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        encoding = self.tokenizer(
            str(row["body_clean"]),
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        labels = []

        for target in self.targets:

            value = row[target]

            if pd.isna(value):
                labels.append(-100)
            else:
                labels.append(
                    int(
                        self.target_encoders[target]
                        .transform([value])[0]
                    )
                )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.long)
        }


In [73]:
recommendation_criterion = nn.CrossEntropyLoss(
    ignore_index=-100
)

rating_criterion = nn.CrossEntropyLoss(
    ignore_index=-100
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

EPOCHS = 3

In [74]:
def calculate_multitask_loss(
    recommendation_logits,
    rating_logits,
    labels
):
    recommendation_labels = labels[:, 0]
    rating_labels = labels[:, 1]

    rec_loss = (
        recommendation_criterion(
            recommendation_logits,
            recommendation_labels
        )
        if (recommendation_labels != -100).any()
        else recommendation_logits.sum() * 0
    )

    rating_loss = (
        rating_criterion(
            rating_logits,
            rating_labels
        )
        if (rating_labels != -100).any()
        else rating_logits.sum() * 0
    )

    total_loss = rec_loss + rating_loss

    return total_loss, rec_loss, rating_loss


In [75]:
# print("Missing recommendation_status:",
#       train_df["recommendation_status"].isna().sum())

# print("Missing rating_band:",
#       train_df["rating_band"].isna().sum())

In [78]:
from tqdm import tqdm


def train_one_epoch(model, loader, optimizer, device):

    model.train()

    total_loss = 0
    total_rec_loss = 0
    total_rating_loss = 0

    for batch in tqdm(loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        recommendation_labels = labels[:, 0]
        rating_labels = labels[:, 1]

        optimizer.zero_grad(set_to_none=True)

        recommendation_logits, rating_logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        rec_loss = recommendation_criterion(
            recommendation_logits,
            recommendation_labels
        )

        rating_loss = rating_criterion(
            rating_logits,
            rating_labels
        )

        loss = rec_loss + rating_loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        total_rec_loss += rec_loss.item()
        total_rating_loss += rating_loss.item()

    n = len(loader)

    return {
        "loss": total_loss / n,
        "recommendation_loss": total_rec_loss / n,
        "rating_loss": total_rating_loss / n
    }


In [79]:
history = []

for epoch in range(EPOCHS):

    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device
    )

    history.append(train_metrics)

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Total Loss: {train_metrics['loss']:.4f}")
    print(
        f"Recommendation Loss: "
        f"{train_metrics['recommendation_loss']:.4f}"
    )
    print(
        f"Rating Loss: "
        f"{train_metrics['rating_loss']:.4f}"
    )


  0%|          | 1067/1878211 [02:42<79:33:54,  6.55it/s]


KeyboardInterrupt: 